<div style="border: 3px solid #b42318; background: #fef3f2; color: #7a271a; border-radius: 12px; padding: 16px 20px; line-height: 1.5">
<div style="font-size: 20px; font-weight: 800; color: #b42318; margin-bottom: 10px">
&#9888;&#65039; ЧЕРНОВИК &mdash; НЕ АКТУАЛЬНАЯ ВЕРСИЯ
</div>
<p style="margin: 0 0 10px 0"><b>Это занятие ещё в работе и будет переписано.</b>
Формулировки, данные и порядок заданий изменятся; часть материала может
опираться на то, что к этому моменту курса ещё не прочитано.</p>
<p style="margin: 0">Заниматься по нему пока не нужно &mdash; дождитесь
окончательной версии. Актуально сейчас только <b>занятие&nbsp;1</b>.</p>
</div>

# Домашняя работа 5. Скользящий контроль изнутри и VC-размерность

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| К лабораторной | занятие 5 — Переобучение, смещение–разброс и скользящий контроль |
| Опора | материал семинара 5 и лекций до него |
| Ожидаемое время | 3–4 часа |
| Данные | **ваша индивидуальная таблица** (по ФИО) |

На занятии мы пользовались `cross_val_score` и `GridSearchCV` как чёрными ящиками. Дома напишем контроль сами, воспроизведём численный пример из конспекта, измерим смещение «лучшего значения по сетке» и проверим перебором, чему равна VC-размерность линейного классификатора.

Свою реализацию контроля вы примените к **своей** выборке и сверите со `sklearn`; эксперименты про смещение и VC-размерность идут на контролируемых данных — там нужен заранее известный ответ.

> **Чем это отличается от занятия.** На семинаре данные были учебные и общие —
> так удобно разбирать. Дома данные ваши: таблица порождается по ФИО, и ни у
> кого в группе она не повторяется. Приёмы те же, числа другие — поэтому
> отвечать придётся за свои числа, а не за преподавательские.


## Как устроена работа

Работа делится на две части, и делятся они по назначению, а не по сложности.

**Обязательная часть — допуск.** Без неё работа не принимается: это тот минимум,
без которого занятие считается неусвоенным. Здесь всегда есть хотя бы одна
реализация «с нуля», сверенная с `scikit-learn` численно.

**Часть на оценку** (помечена значком ★). Она не нужна для допуска — но балл
за работу выставляется именно по ней, и каждый выполненный пункт идёт в зачёт
отдельно. Браться стоит даже за один пункт: это лучше, чем не браться вовсе.


## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO` в обязательной части, код исполняется сверху
   вниз без ошибок в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами,
   со ссылкой на полученные числа;
3. графики подписаны: заголовок, оси, легенда;
4. в ячейке варианта вписано ваше ФИО.

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/ml_labs/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant, submission_name  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from labdata import load_personal
from scipy import optimize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

Регистр, лишние пробелы и написание «ё»/«е» роли не играют. Если ФИО вписано неверно, вариант будет чужим — проверьте вывод ячейки.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=5)
describe_variant(variant)

print("\nСдавать под именем:", submission_name(STUDENT, lab=5))

In [ ]:
# Ваша выборка (эталонная предобработка занятия 1)
data = load_personal(variant)
X_tr, X_te = data["X_train"], data["X_test"]
y_tr, y_te = data["y_train"], data["y_test"]
print(f"{data['domain']}: задача {data['task']}, "
      f"обучающая {X_tr.shape}, контрольная {X_te.shape}")

---
# Задача 1. Скользящий контроль своими руками

Начнём с точного воспроизведения примера 4.9 из конспекта: $y = (2,4,6,8)$,
модель — константа, метод А предсказывает выборочное среднее, метод Б всегда
выдаёт $0$. Лекция даёт $\mathrm{LOO}(A)\approx8.89$ и $\mathrm{LOO}(Б)=30$.

> **Напоминание — LOO.** Leave-one-out, контроль с исключением по одному: предельный случай
> $q$-кратного контроля при $q = \ell$. Каждый объект по очереди изымается,
> модель обучается на оставшихся $\ell-1$, и ошибка считается на изъятом:
>
> $$
> \mathrm{LOO}(\mu, X^\ell) = \frac1\ell\sum_{i=1}^{\ell}
> \mathcal L\bigl(\mu(X^\ell\setminus x_i),\, x_i\bigr).
> $$
>
> Обучающая часть максимально близка к полной выборке, поэтому смещение оценки
> минимально — ценой $\ell$ обучений вместо 5–10 (это мы измерили на семинаре).

In [ ]:
def loo(method, y):
    """Контроль по отдельным объектам: L обучений на L-1 объекте.

    method -- функция, которая по оставшимся ответам выдаёт прогноз.
    Возвращает (среднюю ошибку, список ошибок по объектам).
    """
    raise NotImplementedError


y_toy = np.array([2.0, 4.0, 6.0, 8.0])
# TODO: воспроизведите таблицу примера 4.9 и оба значения LOO
#       (метод А -- np.mean, метод Б -- всегда 0).

### Задание 1.2. Своя реализация $q$-кратного контроля

Реализуйте `my_kfold(n, q, shuffle, seed)`, возвращающий список пар индексов
`(train_idx, test_idx)`, и `my_cross_val_score` поверх него. Сверьте разбиения
со `sklearn.model_selection.KFold` при `shuffle=False` — они обязаны совпасть.

In [ ]:
def my_kfold(n, q, shuffle=True, seed=0):
    """q непересекающихся блоков; возвращает пары (train_idx, test_idx).

    Подсказка: np.array_split делит массив индексов на почти равные части.
    """
    raise NotImplementedError


def my_cross_val_score(make_estimator, X, y, q=5, seed=0):
    """Средняя квадратичная ошибка по блокам."""
    raise NotImplementedError


# TODO: сверьте своё разбиение со sklearn KFold при shuffle=False.

### Задание 1.3. Свой контроль на своей выборке

Примените `my_cross_val_score` к вашей матрице и сверьте со
`sklearn.model_selection.cross_val_score`, передав ему **ваши же** разбиения
через `cv=my_kfold(...)`. При одинаковых разбиениях числа обязаны совпасть до
машинной точности — если нет, ошибка в вашем `my_kfold`.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge

# TODO: 1) постройте разбиения my_kfold(len(y_tr), q=5, shuffle=True, seed=RANDOM_STATE);
#       2) посчитайте my_cross_val_score для Ridge(alpha=1.0) на (X_tr, y_tr);
#       3) передайте ТЕ ЖЕ разбиения в cross_val_score(cv=splits,
#          scoring="neg_mean_squared_error") и сравните поблочно;
#       4) выведите max|своя - sklearn| и оценку со стандартным отклонением.

> **Вывод.** Совпали ли оценки поблочно? Насколько велик разброс между блоками и о чём он говорит?
>
> *(ваш ответ здесь)*

---

> ### ★ Дальше — часть на оценку
>
> Обязательная часть закончилась: если вы дошли досюда и всё работает, работа
> будет принята. Дальше идут задания, по которым выставляется балл. Каждый
> пункт засчитывается отдельно, поэтому имеет смысл сделать хотя бы один.

# Задача 2★. Вложенный контроль

Если по скользящему контролю **выбран** гиперпараметр, то само значение
$\min_\lambda\mathrm{CV}(\mu_\lambda)$ — уже не честная оценка риска.
Лечение — вложенный контроль: внешний цикл оценивает качество, внутренний
подбирает гиперпараметр, и внутренний цикл не видит внешнего блока.

Измерим смещение на **чистом шуме**, где истинное AUC заведомо равно 0.5.

> **Напоминание — парный критерий и $p$-значение.** Мы сравниваем две оценки, полученные **на одних и тех же** данных. Если считать
> их независимыми выборками, общий шум (какие именно данные попались в этом
> повторении) войдёт в разброс обеих и «съест» эффект. Поэтому берут **парную**
> разность $d_i = \text{наивно}_i - \text{вложенно}_i$ по каждому повторению:
> общий шум сокращается, и остаётся только интересующая нас разница.
>
> $p$-значение — вероятность увидеть отличие не меньше наблюдаемого, **если бы**
> разницы на самом деле не было. Малое $p$ означает «случайностью это объяснить
> трудно», и ничего больше: оно не говорит, что эффект велик или важен. Размер
> эффекта — это само среднее $\overline d$ вместе с его погрешностью
> $s_d/\sqrt{N}$. Для парного сравнения: `scipy.stats.ttest_rel`.

In [ ]:
N_TRIALS, n_o, n_f = 30, 80, 100
grid = {"C": np.logspace(-4, 2, 10)}
inner = StratifiedKFold(4, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)

# TODO: N_TRIALS раз сгенерируйте ЧИСТЫЙ ШУМ (X случайный, y случайный)
#       и посчитайте две оценки:
#   naive  -- GridSearchCV по сетке C с внешним контролем, взять best_score_;
#   nested -- внешний контроль для оценки, а внутри каждого блока свой
#             GridSearchCV с inner для подбора C.
naive_arr, nested_arr = ..., ...

In [ ]:
from scipy import stats

# TODO: 1) сравните средние обеих оценок с истиной 0.5 (со стандартной ошибкой);
#       2) посчитайте ПАРНУЮ разность (наивно - вложенно), её стандартную ошибку
#          и парный критерий stats.ttest_rel;
#       3) постройте boxplot обеих оценок с линией 0.5.

> **Вывод.** На данных без всякой связи истинное AUC равно 0.5. Что показали обе оценки? Почему парная разность разрешается статистически, а абсолютные значения — нет?
>
> *(ваш ответ здесь)*

---
# Задача 3. VC-размерность перебором

Определение 4.12: $\mathrm{VCdim}(A)$ — наибольшее $d$, для которого существует
набор из $d$ точек, разбиваемый моделью $A$, то есть реализуются **все** $2^d$
разметок. Пример 4.13: для линейных классификаторов в $\mathbb{R}^2$ она равна 3.

Проверим перебором. Линейная разделимость набора $(X,y)$ — это разрешимость
системы $y_i(w^{\mathsf T}x_i + b)\ge1$, то есть задача линейного
программирования с нулевой целевой функцией: важна только совместность.

In [ ]:
from itertools import product


def linearly_separable(X, y):
    """Существуют ли w, b с y_i (w^T x_i + b) >= 1?

    Подсказка: задача ЛП с нулевой целевой функцией. Ограничения приводятся
    к виду A_ub @ z <= b_ub, где z = (w, b). Совместность -> res.status == 0.
    """
    raise NotImplementedError


def shatters(X):
    """Реализуются ли все 2^d разметок? Верните (да/нет, нереализуемая разметка)."""
    raise NotImplementedError


# TODO: проверьте, что 3 точки общего положения разбиваются, а 4 -- нет
#       (и для квадрата, и для случая «точка внутри треугольника»);
#       выведите конкретную нереализуемую разметку в каждом случае.

In [ ]:
# TODO: для n = 2, 3, 5 и d = 2..8 посчитайте долю реализуемых разметок
#       (усредните по 15 случайным наборам точек), сведите в таблицу
#       и постройте график с вертикальными отметками d = n + 1.

> **Вывод.** Какая разметка четырёх точек оказалась нереализуемой в каждом случае? Где ломается кривая доли разделимых разметок и как это связано с $\mathrm{VCdim} = n+1$?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Почему по обучающей ошибке нельзя выбрать сложность модели? Сошлитесь на вложенность моделей.
2. Вам нужно сравнить две модели и заявить, что одна лучше. Опишите протокол в четырёх пунктах так, чтобы в нём не было ни одной из разобранных ловушек.

## Обратная связь

Это не оценивается и на балл не влияет — нужно, чтобы поправить работу к
следующему году. Отвечайте одной строкой, честно.

| | |
|---|---|
| Сколько часов заняло | |
| Сложность от 1 до 5 | |
| Что осталось непонятным | |
| Какое задание показалось лишним | |

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.

Имя файла — то, что напечатала ячейка с вариантом: `hw01_Ivanov_I_I.ipynb`.
Номер работы впереди, фамилия и инициалы латиницей. Если сдаёте исправленную
версию, допишите `_v2`.